<a href="https://colab.research.google.com/github/hitarthi45/Artificial-Intelligence/blob/main/(EXP_10)Text_rank_summarisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TextRank Document summarisation

In [2]:
#import libraries
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import networkx as nx

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
def preprocess_text(text):
  stop_words = set(stopwords.words('english'))
  sentences = sent_tokenize(text)
  all_word_frequencies = []

  for sent in sentences:
    word = word_tokenize(sent)
    # Convert words to lowercase for consistent counting
    filtered_words = [w.lower() for w in word if w.isalnum() and w.lower() not in stop_words]
    all_word_frequencies.append(Counter(filtered_words))

  return sentences, all_word_frequencies

In [5]:
def similarity_matrix(word_frequencies_list):
  size = len(word_frequencies_list)
  sm_matrix = np.zeros((size, size))
  for i in range(size):
    for j in range(size):
      if i != j:
        words1 = word_frequencies_list[i]
        words2 = word_frequencies_list[j]
        common_words = set(words1.keys()).union(set(words2.keys()))

        # Handle cases where there are no common words (vectors would be zero)
        if not common_words:
            sm_matrix[i][j] = 0.0
            continue

        vec1 = np.array([words1.get(word, 0) for word in common_words])
        vec2 = np.array([words2.get(word, 0) for word in common_words])

        # cosine_similarity handles zero vectors by returning 0
        sm_matrix[i][j] = cosine_similarity([vec1], [vec2])[0, 0]
  return sm_matrix

In [6]:
def textrank(text,num=3):
  sentences,word_frequencies=preprocess_text(text)
  sm=similarity_matrix(word_frequencies)

  #build graph and apply pagerank
  graph=nx.from_numpy_array(sm)
  scores=nx.pagerank(graph)

  ranked_sentences=sorted(((scores.get(i,0),s)
            for i,s in enumerate(sentences)),reverse=True)
  summary=" ".join(sent for _,sent in ranked_sentences[:num])
  return summary



In [7]:
text = """ India is situated north of the equator between 8°4' north (the mainland) to 37°6' north latitude and 68°7' east to 97°25' east longitude.[2] It is the seventh-largest country in the world, with a total area of 3,287,263 square kilometres (1,269,219 mi2).[3][4][5] India measures 3,214 km (1,997 mi) from north to south and 2,933 km (1,822 mi) from east to west. It has a land frontier of 15,200 km (9,445 mi) and a coastline of 7,516.6 km (4,671 mi).[1]

On the south, India projects into and is bounded by the Indian Ocean—in particular, by the Arabian Sea on the west, the Lakshadweep Sea to the southwest, the Bay of Bengal on the east, and the Indian Ocean proper to the south. The Palk Strait and Gulf of Mannar separate India from Sri Lanka to its immediate southeast, and the Maldives are some 125 kilometres (78 mi) to the south of India's Lakshadweep Islands across the Eight Degree Channel. India's Andaman and Nicobar Islands, some 1,200 kilometres (750 mi) southeast of the mainland, share maritime borders with Myanmar, Thailand and Indonesia. The southernmost tip of the Indian mainland (8°4′38″N, 77°31′56″E) is just south of Kanyakumari, while the southernmost point in India is Indira Point on Great Nicobar Island. The northernmost point which is under Indian administration is Indira Col, Siachen Glacier.[6] India's territorial waters extend into the sea to a distance of 12 nautical miles (13.8 mi; 22.2 km) from the coast baseline.[7] India has the 18th largest Exclusive Economic Zone of 2,305,143 km2 (890,021 mi2).

The northern frontiers of India are defined largely by the Himalayan mountain range, where the country borders China, Bhutan, and Nepal. Its western border with Pakistan lies in the Karakoram and Western Himalayan ranges, Punjab Plains, the Thar Desert and the Rann of Kutch salt marshes. In the far northeast, the Chin Hills and Kachin Hills, deeply forested mountainous regions, separate India from Myanmar. On the east, its border with Bangladesh is largely defined by the Khasi Hills and Mizo Hills, and the watershed region of the Indo-Gangetic Plain.[clarification needed] """

In [8]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [9]:
print(textrank(text))

[3][4][5] India measures 3,214 km (1,997 mi) from north to south and 2,933 km (1,822 mi) from east to west. The Palk Strait and Gulf of Mannar separate India from Sri Lanka to its immediate southeast, and the Maldives are some 125 kilometres (78 mi) to the south of India's Lakshadweep Islands across the Eight Degree Channel. India's Andaman and Nicobar Islands, some 1,200 kilometres (750 mi) southeast of the mainland, share maritime borders with Myanmar, Thailand and Indonesia.


###POST-LAB

In [1]:
my_text = """ I am Hitarthi Pansuriya. I am 20 years old.
              I am currently pursuing B.Tech. in ICT engineering.
              I want to become a software developer and I am seeing myself as a senior developer in next 5-6 years.
              Maybe my starting company will be a small firm but I will make sure to level up to MNC company """

In [10]:
from nltk.tokenize import sent_tokenize

# Assuming 'text' variable from previous cells is the portfolio
original_sentences = sent_tokenize(my_text)
total_sentences = len(original_sentences)

summary_50_percent_len = int(total_sentences * 0.50)
summary_25_percent_len = int(total_sentences * 0.25)

print(f"Original text has {total_sentences} sentences.")
print(f"50% summary will have approximately {summary_50_percent_len} sentences.")
print(f"25% summary will have approximately {summary_25_percent_len} sentences.\n")

summary_50 = textrank(my_text, num=summary_50_percent_len)
summary_25 = textrank(my_text, num=summary_25_percent_len)

print("### 50% Summary:")
print(summary_50)
print("\n### 25% Summary:")
print(summary_25)

Original text has 6 sentences.
50% summary will have approximately 3 sentences.
25% summary will have approximately 1 sentences.

### 50% Summary:
I want to become a software developer and I am seeing myself as a senior developer in next 5-6 years. I am 20 years old. in ICT engineering.

### 25% Summary:
I want to become a software developer and I am seeing myself as a senior developer in next 5-6 years.
